# INSTALL

In [1]:
!pip install typhoon-ocr pdf2image Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 19.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 32.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21/21 [typhoon-ocr] [openai]c]


In [2]:
!apt-get install -y poppler-utils

zsh:1: command not found: apt-get


# IMPORT

In [ ]:
import os
import json
import re
import requests
from typhoon_ocr import ocr_document
import google.generativeai as genai

# Pipeline

In [ ]:
class Extractor:
    #  Initialize
    def __init__(self, api_key: str):
        """Initializes the OCR engine with the provided API key."""
        # OCR
        os.environ["TYPHOON_OCR_API_KEY"] = api_key
        # Parse
        genai.configure(api_key=gemini_api_key)
        self.model = genai.GenerativeModel('gemini-1.5-flash')
    #  Image to dict
    def extract_pdf(self, pdf_path: str) -> str:
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"The file {pdf_path} was not found.")
        
        print(f"Converting PDF with Typhoon OCR: {pdf_path}")

        raw_markdown = ocr_document(pdf_path)

        return self._parse_markdown_byChat(raw_markdown)
    # Text to Dict
    def _parse_markdown_byChat(self, full_text: str) -> str:
        # PROMPT
        prompt = f"""
        Extract the election results from the following Thai OCR text into a structured JSON format.
        
        Requirements:
        1. Fix any OCR typos in province, district, or party names.
        2. Convert all numbers (including Thai digits) to standard integers.
        3. Include a 'metadata' section (province, district, unit), a 'summary' section (ballot counts), 
           and a 'results' list (party number, name, and votes).
        4. check the number of score they got and and thai text in () if not turn thai text into integer and use that
        OCR Text:
        {full_text}
        """

        response = self.model.generate_content(
            prompt,
            generation_config={"response_mime_type": "application/json"}
        )
        return response.text



        